# Connecting to Hardware

## Connecting to ChipWhisperer

Now that your hardware is all setup, we can now learn how to connect to it. We can connect to the ChipWhisperer with:

In [29]:
SCOPETYPE = 'OPENADC'
PLATFORM = 'CWHUSKY'
#CRYPTO_TARGET = 'TINYAES128C'
CRYPTO_TARGET = 'None'
SS_VER = 'SS_VER_1_1'

In [30]:
import chipwhisperer as cw

if "scope" in globals() and scope is not None:
    try:
        scope.dis()
    except Exception:
        pass

if "target" in globals() and target is not None:
    try:
        target.dis()
    except Exception:
        pass

scope = cw.scope()
        
target = cw.target(scope, cw.targets.SimpleSerial) #cw.targets.SimpleSerial can be omitted

scope.default_setup()

scope.gain.mode                          changed from low                       to high                     
scope.gain.gain                          changed from 0                         to 22                       
scope.gain.db                            changed from 15.0                      to 25.091743119266056       
scope.adc.samples                        changed from 346246                    to 5000                     
scope.clock.clkgen_freq                  changed from 0                         to 7363636.363636363        
scope.clock.adc_freq                     changed from 0                         to 29454545.454545453       
scope.clock.adc_rate                     changed from 0.0                       to 29454545.454545453       
scope.io.tio1                            changed from serial_tx                 to serial_rx                
scope.io.tio2                            changed from serial_rx                 to serial_tx                
scope.io.hs2       

In [21]:
%%bash
cd ../firmware/mcu/simpleserial-base/
make PLATFORM=CWHUSKY CRYPTO_TARGET=NONE SS_VER=SS_VER_1_1

SS_VER set to SS_VER_1_1
SS_VER set to SS_VER_1_1
.
Welcome to another exciting ChipWhisperer target build!!
arm-none-eabi-gcc (15:14.2.rel1-1) 14.2.1 20241119
Copyright (C) 2024 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.

mkdir -p objdir-CWHUSKY 
.
Compiling:
-en     simpleserial-base.c ...
-e Done!
.
Compiling:
-en     .././simpleserial/simpleserial.c ...
-e Done!
.
Compiling:
-en     .././hal/hal.c ...
-e Done!
.
Compiling:
-en     .././hal//sam4s/startup_sam4s.c ...
-e Done!
.
Compiling:
-en     .././hal//sam4s/sam4s_hal.c ...
-e Done!
.
Compiling:
-en     .././hal//sam4s/uart.c ...
-e Done!
.
Compiling:
-en     .././hal//sam4s/pio.c ...
-e Done!
.
Compiling:
-en     .././hal//sam4s/system_sam4s.c ...
-e Done!
.
Compiling:
-en     .././hal//sam4s/sysclk.c ...
-e Done!
.
Compiling:
-en     .././hal//sam4s/pmc.c ...
-e Done!
.
LINKING:
-en     sim

In [4]:
#cw.program_target(scope, cw.programmers.XMEGAProgrammer, "path/to/firmware.hex")
#cw.program_target(scope, cw.programmers.STM32FProgrammer, "path/to/firmware.hex")
#cw.program_target(scope, cw.programmers.AVRProgrammer, "path/to/firmware.hex")
#cw.program_target(scope, cw.programmers.SAM4SProgrammer, "path/to/firmware.hex")
cw.program_target(scope, cw.programmers.SAM4SProgrammer, "../firmware/mcu/simpleserial-aes/simpleserial-aes-CWHUSKY.hex")

## Communication with the Target

Communication with targets, which is done through the `SimpleSerial target` object we got earlier, is grouped into two categories:

1. Raw serial via `target.read()`, `target.write()`, `target.flush()`, etc. 

1. SimpleSerial commands via `target.simpleserial_read()`, `target.simpleserial_write()`, `target.simpleserial_wait_ack()`, etc.

The firmware we uploaded uses the simpleserial protocol (https://wiki.newae.com/SimpleSerial), so we'll start off with simpleserial. Later, we'll use the raw serial commands to send the same messages.

If you check the simpleserial-base firmware (`simpleserial-base.c`) you'll find that for the simpleserial `'p'` command, the target will echo back the command. Let's try that out now:

In [31]:
print(scope.io)
print(scope.clock)
print(target)

tio1            = serial_rx
tio2            = serial_tx
tio3            = high_z
tio4            = high_z
pdid            = high_z
pdic            = high_z
nrst            = high_z
glitch_hp       = False
glitch_lp       = False
extclk_src      = hs1
hs2             = clkgen
target_pwr      = True
tio_states      = (1, 1, 1, 0)
cdc_settings    = [0, 0, 0, 0]
aux_io_mcx      = high_z
glitch_trig_mcx = trigger

clkgen_src             = system
clkgen_freq            = 7363636.363636363
adc_mul                = 4
adc_freq               = 29454545.454545453
adc_rate               = 29454545.454545453
freq_ctr               = 0
freq_ctr_src           = extclk
clkgen_locked          = True
adc_phase              = 0.0
extclk_monitor_enabled = False
extclk_error           = False
extclk_tolerance       = 1144409.1796875

SimpleSerial Settings =
	output_len             = 16
	baud                   = 38400
	simpleserial_last_read = 
	simpleserial_last_sent = 
	xonxoff                = None
	curr

In [43]:
print(target)
print("baud:", target.baud)

target.flush()

target.simpleserial_write('v', bytearray())

import time
time.sleep(0.1)

print("bytes waiting:", target.in_waiting())
print("raw response:", repr(target.read()))

SimpleSerial Settings =
	output_len             = 16
	baud                   = 38400
	simpleserial_last_read = z01
	
	simpleserial_last_sent = v
	
	xonxoff                = None
	currently_xoff         = None
	parity                 = none
	stop_bits              = 1
	
baud: 38400
bytes waiting: 4
raw response: 'z01\n'


expected output
```
SimpleSerial Settings =
	output_len             = 16
	baud                   = 38400
	simpleserial_last_read = z01
	
	simpleserial_last_sent = v
	
	xonxoff                = None
	currently_xoff         = None
	parity                 = none
	stop_bits              = 1
	
baud: 38400
bytes waiting: 4
raw response: 'z01\n'
```

In [47]:
# clear receive buffer
target.flush()
msg = bytearray([0]*16) #simpleserial uses bytearrays
print("TX -> SAM4S:", msg.hex())
target.simpleserial_write('p', msg)

TX -> SAM4S: 00000000000000000000000000000000


Let's check if we got a response:

In [48]:
print(target.simpleserial_read('r', 16))

bytearray(b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00')


In [58]:
# clear receive buffer
target.flush()
msg = bytearray([0]*16) #simpleserial uses bytearrays
print("TX -> SAM4S:", msg.hex())
target.simpleserial_write('p', msg)
print(target.simpleserial_read('r', 16))

TX -> SAM4S: 00000000000000000000000000000000
bytearray(b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00')


It also has a `'k'` command. Try sending that now:

In [59]:
msg = bytearray([0]*16) #simpleserial uses bytearrays
print("TX -> SAM4S:", msg.hex())
target.simpleserial_write('k', msg)

TX -> SAM4S: 00000000000000000000000000000000


That command doesn't return anything to us, but it should ack and give us an error return:

In [60]:
print(target.simpleserial_wait_ack()) #should return 0

0


In [67]:
msg = bytearray([0]*16) #simpleserial uses bytearrays
print("TX -> SAM4S:", msg.hex())
target.simpleserial_write('k', msg)
print(target.simpleserial_wait_ack()) #should return 0

TX -> SAM4S: 00000000000000000000000000000000
0


Simpleserial messages generally take the form:

```python
command_character + ascii_encoded_bytes + '\n'
```

For our first command, `command_character='p'` and `ascii_encoded_bytes="00"*16` (keep in mind this isn't a binary `0x00`, it's ASCII `"00"`, which has a binary value of `0x3030`). Try resending the `'p'` command from earlier using `target.write()`:

In [103]:
msg = 'p' + "00112233445566778899AABBCCDDEEFF" + "\n"
print("TX -> SAM4S:", msg.encode().hex())
target.write(msg) 

TX -> SAM4S: 7030303131323233333434353536363737383839394141424243434444454546460a


A simple `target.read()` will return all the characters that have been sent back from the target so far. Let's see what the device returned to us:

In [101]:
recv_msg = ""

In [104]:
recv_msg += target.read() #you might have to run this block a few times to get the full message
print(recv_msg)

r00112233445566778899AABBCCDDEEFF
z00
r00112233445566778899AABBCCDDEEFF
z00



The simpleserial commands are usually sufficient for taking to simpleserial firmware, but you'll need the raw serial commands for some of the other labs.

## SimpleSerial 2

As of ChipWhisperer 5.4, a new target communication protocol, SimpleSerial 2 is available. It has a number of advantages over the old SimpleSerial protocol:

1. Data is unencoded binary (except for byte stuffing) instead of ACII encoded, meaning far more data can be sent per packet (just under double)
1. It's got a command and subcommand field that get passed to the target callback functions, meaning more can be done per packet. For example, the simpleserial-aes firmware has been modified to allow plaintext, key, and/or mask to be set with a single packet.
1. It's got a length field, meaning the same target commands can take different length packets depending on the situation. This, for example, allows both plaintext and key (or only one) to be send with the same command in simpleserial-aes.
1. It's got an 8-bit CRC (0xA6) for data integrety.
1. Frames are consistant overhead byte stuffed (COBS). This makes it easy to reset communication (done simply by sending two 0x00 bytes) and helps catch malformed length bytes.

It is, however, a more complicated protocol. You can use it by compiling firmware with `SS_VER=SS_VER_2_0` and using `cw.targets.SimpleSerial2`. In `sca101` and `fault101`, it can be used by setting `SS_VER='SS_VER_2_0'` when setting `PLATFORM` and `SCOPETYPE`.

## Capturing Traces

Now that the target's programmed and we know how to communicate with it, let's start recording some power traces! To capture a trace:

1. Arm the ChipWhisperer with `scope.arm()`. It will begin capturing as soon as it is triggered (which in our case is a rising edge on `gpio4`.
1. `scope.capture()` will read back the captured power trace, blocking until either ChipWhisperer is done recording, or the scope times out. Note that the error return will tell you whether or not the scope timed out. It does not return the captured scope data.
1. You can read back the captured power trace with `scope.get_last_trace()`.

`simpleserial_base` will trigger the ChipWhisperer when we send the `'p'` command. Try capturing a trace now:

In [105]:
msg = bytearray([0]*16)
print("arm")
scope.arm()
print("running")
target.simpleserial_write('p', msg)  # p do trigger
print("ran")
scope.capture()
print("capture done")
print(scope.get_last_trace())

arm
running
ran
capture done
[1607  590 1505 ... 2279 2493 2599]


ChipWhisperer also has a `capture_trace()` convience function that:

1. Optionally sends the `'k'` command
1. Arms the scope
1. Sends the `'p'` command
1. Captures the trace
1. Reads the return `'r'` message
1. Returns a `Trace` class that groups the trace data, `'p'` message, the `'r'` message, and the `'k'` message

It isn't always the best option to use, but it's usually sufficient for most simpleserial applications

## Conclusion

And that's it! You should be all ready to continue on to SCA101 (or any other of our courses).

We've glossed over some stuff here, so consult the [API documentation](https://chipwhisperer.readthedocs.io/en/latest/scope-api.html) or ask on our [forums](https://forum.newae.com) if you get stuck.

As a final step, we should disconnect from the hardware so it doesn't stay "in use" by this notebook.

In [106]:
scope.dis()
target.dis()

## ChipWhisperer-Husky

While the ChipWhisperer-Husky has many new features, its API has been designed to keep most basic tasks the same, especially with power analysis. You should be able to go through SCA101 without learning anything Husky specific and fault101 will cover any Husky specific things you need to know. That being said, you may still find it helpful to refer to the `demos/husky` folder, starting with [01 - Introduction to ChipWhisperer-Husky.ipynb](demos/husky/01%20-%20Introduction%20to%20ChipWhisperer-Husky.ipynb) to learn more about its advanced features.